# zero-shot with marker genes

In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
# Import necessary libraries here

## data preprocess

In [ ]:
# Data preprocessing code here

# one-shot with marker genes

# zero-shot with deconv

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import adjusted_rand_score

import openai
import ast
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

from utils import *
from prompt import *
import prompt


## data preprocess

In [ ]:
data_name = "151673"
name_truth = "layer_guess"
# --- Load data ---
data_path = str(dataset_dir("visium_libd", data_name))
# marker_path = "data/reference/marker_genes/Mouse_cell_markers.txt"
adata = sc.read_visium(data_path)
adata.var_names_make_unique()

# Normalize data
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)

cell_proportion_data = pd.read_csv(os.path.join(data_path, f"celltype_proportions_{data_name}.csv"), index_col=0)

adata.obs = adata.obs.join(cell_proportion_data)

# read the metadata
meta_data = pd.read_csv(os.path.join(data_path, "metadata.tsv"), sep="\t")
# merge the metadata to adata
adata.obs = adata.obs.merge(meta_data, left_index=True, right_index=True, how="left")

# Remove rows with NaN values in 'layer_guess'
adata = adata[~adata.obs[name_truth].isna()].copy()

# Verify that NaNs have been removed
remaining_nan_count = adata.obs[name_truth].isna().sum()
print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

# Rename the obs_names of adata
adata.obs_names = [f'spot_{i}' for i in range(len(adata.obs_names))]
# Transform spatial coordinates to DataFrame for sparse_adjacency
pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
cell_proportion_data = adata.obs[cell_proportion_data.columns].copy()

r = 200
adj_matrix, distances = sparse_adjacency(pos_data, threshold=r, add_diagonal=True)
# Calculate the number of neighbors of each node
n_neighbors = adj_matrix.sum(axis=1).mean()
print(f"Number of neighbors: {sig_figs(n_neighbors, 3)}")

# If you want to store the n_neighbors of each node
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
adata.obs['n_neighbors'] = n_neighbors


In [ ]:
# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(cell_proportion_data)


# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

# transform the neighbor matrix to a dataframe
neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized, index=cell_proportion_data.index, 
                              columns=cell_proportion_data.columns)



In [ ]:
neighbor_normalized_df

## plot deconvolution results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate mean cell type proportions for each group
mean_proportions = cell_proportion_data.groupby(adata.obs[name_truth]).mean()


# Create a stacked bar plot
fig, ax = plt.subplots(figsize=(10, 8))
mean_proportions.plot(kind='bar', stacked=True, ax=ax)

# Customize the plot
plt.title('Cell Type Proportions by Layer', fontsize=16)
plt.xlabel('Layer', fontsize=12)
plt.ylabel('Mean Proportion', fontsize=12)
plt.legend(title='Cell Type', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=14)
plt.tight_layout()

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')

# Add percentage labels on the bars
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f', label_type='center', fontsize=8)

plt.show()

# Display the mean proportions as a table
print(mean_proportions.to_string())




In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as colors

# Create a custom normalization
norm = colors.TwoSlopeNorm(vmin=0, vcenter=0.1, vmax=0.3)

# Create a heatmap with the viridis colormap and custom normalization
plt.figure(figsize=(10, 8))
sns.heatmap(mean_proportions, annot=True, cmap='viridis', norm=norm, fmt='.2f', 
            cbar_kws={'label': 'Mean Proportion'})

plt.title('Heatmap of Cell Type Proportions by Layer', fontsize=16)
plt.xlabel('Cell Type', fontsize=12)
plt.ylabel('Layer', fontsize=12)
plt.tight_layout()
plt.show()

## prompt

In [ ]:
# Generate a dictionary mapping unique values in 'layer_guess' to integer keys starting from 0
unique_layers = adata.obs[name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

cell_names_mapping ={'Astro': 'Astrocyte',
 'EndoMural': 'Endothelial and mural cells',
 'Excit_L2_3': 'Excitatory neuron layer 2/3',
 'Excit_L3': 'Excitatory neuron layer 3',
 'Excit_L3_4_5': 'Excitatory neuron layer 3/4/5',
 'Excit_L4': 'Excitatory neuron layer 4',
 'Excit_L5': 'Excitatory neuron layer 5',
 'Excit_L5_6': 'Excitatory neuron layer 5/6',
 'Excit_L6': 'Excitatory neuron layer 6',
 'Inhib': 'Inhibitory neuron',
 'Micro': 'Microglia',
 'OPC': 'Oligodendrocyte precursor cell',
 'Oligo': 'Oligodendrocyte'}


config = load_config('configs/config_151673_zeroshot.yaml')
config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping




In [ ]:
print(config)

In [ ]:
x = [i for i in range(len(adata)) if adata.obs[name_truth].iloc[i] == "Layer3"][30:40]
print(prompt.zeroshot_celltype(neighbor_normalized_df, x, config))



## GPT

In [ ]:
generate_json_end2end(neighbor_normalized_df, config, prompt_func=prompt.zeroshot_celltype, n_rows=1, batch_size=5000)

In [ ]:
# submit the json

In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 1
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=extract_dict.keys())], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['zeroshot_gpt4o']


In [ ]:
gpt_results_df.index.difference(neighbor_normalized_df.index)

## Gemini

In [ ]:
import google.generativeai as genai
import pickle
import time



genai.configure(api_key=os.environ["API_KEY"])
model = genai.GenerativeModel("gemini-1.5-pro")
gen_config=genai.types.GenerationConfig(temperature=1.0, max_output_tokens=1000)


In [ ]:
gemini_results_df, store_responses = run_gemini(model, gen_config, neighbor_normalized_df, config, prompt.zeroshot_celltype, n_rows=1, column_name="zeroshot_gemini")

with open(f'./gemini_results/{config.data_name}_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.pkl', 'wb') as file:
    pickle.dump(store_responses, file)

gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


## process output

In [ ]:
gemini_results_df.index.difference(adata.obs_names)

In [ ]:
gemini_results_df.zeroshot_gemini.value_counts()


## plot

In [ ]:
adata.obs = adata.obs.join(gpt_results_df)
# replace the NA in adata.obs['zeroshot_gemini'] with "unknown"
adata.obs['zeroshot_gpt4o'] = adata.obs['zeroshot_gpt4o'].fillna("unknown")

In [ ]:
sc.pl.spatial(adata, color=['zeroshot_gpt4o', name_truth], library_id=data_name, size=1.4)
print(adjusted_rand_score(adata.obs[name_truth], adata.obs['zeroshot_gpt4o']))

# one shot with deconv

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import json
import scipy
from sklearn.model_selection import train_test_split
from sklearn.metrics import adjusted_rand_score

import openai
import ast
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

from utils import *
from prompt import *
import prompt

In [ ]:
import importlib  # Import the importlib module
import prompt
importlib.reload(prompt)  # Reload the module to get the updated functions

## data preprocess

In [ ]:
config = load_config('configs/config_151509_finetune.yaml')

In [ ]:
data_name = config.data_name
name_truth = config.name_truth
# --- Load data ---
data_path = str(dataset_dir("visium_libd", data_name))
# marker_path = "data/reference/marker_genes/Mouse_cell_markers.txt"
adata = sc.read_visium(data_path)
adata.var_names_make_unique()

# Normalize data
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)

cell_proportion_data = pd.read_csv(os.path.join(data_path, f"celltype_proportions_{data_name}.csv"), index_col=0)

adata.obs = adata.obs.join(cell_proportion_data)

# read the metadata
meta_data = pd.read_csv(os.path.join(data_path, "metadata.tsv"), sep="\t")
# merge the metadata to adata
adata.obs = adata.obs.merge(meta_data, left_index=True, right_index=True, how="left")

# Remove rows with NaN values in 'layer_guess'
adata = adata[~adata.obs[name_truth].isna()].copy()

# Verify that NaNs have been removed
remaining_nan_count = adata.obs[name_truth].isna().sum()
print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

# Rename the obs_names of adata
adata.obs_names = [f'spot_{i}' for i in range(len(adata.obs_names))]

# Transform spatial coordinates to DataFrame for sparse_adjacency
pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
cell_proportion_data = adata.obs[cell_proportion_data.columns].copy()

adj_matrix, distances = sparse_adjacency(pos_data, threshold=config.r, add_diagonal=True)

# Calculate the number of neighbors of each node
n_neighbors = adj_matrix.sum(axis=1).mean()
print(f"Number of neighbors: {sig_figs(n_neighbors, 3)}")

# If you want to store the n_neighbors of each node
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
adata.obs['n_neighbors'] = n_neighbors


In [ ]:
# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(cell_proportion_data)


# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

# transform the neighbor matrix to a dataframe
neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized, index=cell_proportion_data.index, 
                              columns=cell_proportion_data.columns)



In [ ]:
# seperate data into train and val
# 设定随机种子
seed = 42  # 你可以根据需要修改这个值

# 定义分割比例 p (比如 0.7 表示 70% 数据用于训练，30% 数据用于测试)
p = 0.3

# 分割数据集为训练集和测试集
train_neighbor_normalized_df, val_neighbor_normalized_df = train_test_split(neighbor_normalized_df, 
                                                            test_size=1-p, 
                                                            random_state=seed,
                                                            stratify=adata.obs[name_truth]
                                                           )

# check sample distribution
adata.obs[name_truth].loc[train_neighbor_normalized_df.index].value_counts()

In [ ]:
# Mean prototype
one_shot_df = pd.concat([adata.obs[name_truth].loc[train_neighbor_normalized_df.index], train_neighbor_normalized_df], axis=1).groupby(name_truth).mean()
one_shot_df

## prompt

In [ ]:
# Generate a dictionary mapping unique values in 'layer_guess' to integer keys starting from 0
unique_layers = adata.obs[name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

cell_names_mapping ={'Astro': 'Astrocyte',
 'EndoMural': 'Endothelial and mural cells',
 'Excit_L2_3': 'Excitatory neuron layer 2/3',
 'Excit_L3': 'Excitatory neuron layer 3',
 'Excit_L3_4_5': 'Excitatory neuron layer 3/4/5',
 'Excit_L4': 'Excitatory neuron layer 4',
 'Excit_L5': 'Excitatory neuron layer 5',
 'Excit_L5_6': 'Excitatory neuron layer 5/6',
 'Excit_L6': 'Excitatory neuron layer 6',
 'Inhib': 'Inhibitory neuron',
 'Micro': 'Microglia',
 'OPC': 'Oligodendrocyte precursor cell',
 'Oligo': 'Oligodendrocyte'}


config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping

config.oneshot_prompt = prompt.CP_celltype(one_shot_df, config)

In [ ]:

index_to_process = [i for i in val_neighbor_normalized_df.loc[adata.obs[name_truth]=="Layer5"].index]
print(config.oneshot_prompt + prompt.oneshot_celltype(val_neighbor_normalized_df.loc[index_to_process], range(10), config))

In [ ]:
generate_json_end2end(val_neighbor_normalized_df, config, prompt_func=prompt.oneshot_celltype, n_rows=1, batch_size=5000)

## GPT

In [ ]:
# submit_spot.py

In [ ]:
# manually fetch the results
batch_id = "batch_6712a0cf7c2c819092a1b102f967cace"
number = 1
file_response = client.files.content(client.batches.retrieve(batch_id).output_file_id)
save_name = f"response_{config.data_name}_{number}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
output_file_name = f"{config.output_path}/{save_name}"
# Open the file in write mode and save the string
with open(output_file_name, 'w') as file:
    file.write(file_response.text) 



In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 1
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=extract_dict.keys())], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['oneshot_gpt4o']

In [ ]:
gpt_results_df.index.difference(val_neighbor_normalized_df.index)

In [ ]:
gpt_results_df.oneshot_gpt4o.value_counts()

## Gemini

In [ ]:
import google.generativeai as genai
import pickle
import time



genai.configure(api_key=os.environ["API_KEY"])
model = genai.GenerativeModel("gemini-1.5-pro")
gen_config=genai.types.GenerationConfig(temperature=1.0, max_output_tokens=1000)

In [ ]:
gemini_results_df, store_responses = run_gemini(model, gen_config, val_neighbor_normalized_df, 
                                                config, prompt_func=prompt.oneshot_celltype, 
                                                n_rows=1, column_name="oneshot_gemini")

with open(f'./gemini_results/{config.data_name}_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.pkl', 'wb') as file:
    pickle.dump(store_responses, file)

gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
gemini_results_df.index.difference(val_neighbor_normalized_df.index)



## plot

In [ ]:
val_adata = adata[val_neighbor_normalized_df.index].copy()


In [ ]:
val_adata.obs = val_adata.obs.join(gemini_results_df)
val_adata.obs = val_adata.obs.join(gpt_results_df)
val_adata.obs['oneshot_gemini'] = val_adata.obs['oneshot_gemini'].fillna("unknown")
val_adata.obs['oneshot_gpt4o'] = val_adata.obs['oneshot_gpt4o'].fillna("unknown")
sc.pl.spatial(val_adata, color=['oneshot_gpt4o', 'oneshot_gemini', name_truth], library_id=data_name, size=1.6)


In [ ]:
print(adjusted_rand_score(val_adata.obs[name_truth], val_adata.obs['oneshot_gpt4o']))
print(adjusted_rand_score(val_adata.obs[name_truth], val_adata.obs['oneshot_gemini']))


# finetune with deconv

## data preprocess

In [ ]:
# same as one-shot

## prompt

In [ ]:
i = 6
# system prompt
system_p = prompt.finetune_system_deconv(config)
# user prompt
user_p = prompt.finetune_user_deconv(train_neighbor_normalized_df, i, config)
# assistant prompt
assistant_p = prompt.finetune_assistant(train_neighbor_normalized_df, i, adata.obs[name_truth])

In [ ]:
print(system_p)
print(user_p)
print(assistant_p)


In [ ]:
_, val_for_finetune = train_test_split(val_neighbor_normalized_df, 
                                                            test_size=0.1, 
                                                            random_state=seed
                                                           )

adata.obs.loc[val_for_finetune.index, name_truth].value_counts()

In [ ]:
# generate json for finetune
output_folder = f"finetune_json/{config.data_name}_{config.model_type}/"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

train_output_file = f"{output_folder}{config.data_name}_train_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
print(f"Generating json for training into {train_output_file}")
with open(train_output_file, 'w') as f:
    for i in range(train_neighbor_normalized_df.shape[0]):
        system_p = prompt.finetune_system_deconv(config)
        user_p = prompt.finetune_user_deconv(train_neighbor_normalized_df, i, config)
        assistant_p = prompt.finetune_assistant(train_neighbor_normalized_df, i, adata.obs[name_truth])
        
        row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
        json_str = json.dumps(row_data)
        f.write(json_str + '\n')  

val_output_file = f"{output_folder}{config.data_name}_val_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
print(f"Generating json for validation into {val_output_file}")
with open(val_output_file, 'w') as f:
    for i in range(val_for_finetune.shape[0]):
        system_p = prompt.finetune_system_deconv(config)
        user_p = prompt.finetune_user_deconv(val_for_finetune, i, config)
        assistant_p = prompt.finetune_assistant(val_for_finetune, i, adata.obs[name_truth])

        row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
        json_str = json.dumps(row_data)
        f.write(json_str + '\n')  


## submit

In [ ]:

train_file = client.files.create(
  file=open(train_output_file, "rb"),
  purpose="fine-tune"
)

val_file = client.files.create(
  file=open(val_output_file, "rb"),
  purpose="fine-tune"
)

finetune_job = client.fine_tuning.jobs.create(
  training_file=train_file.id,
  validation_file=val_file.id,
  model="gpt-4o-mini-2024-07-18",
  suffix="151509_deconv_with_numbers"   # default is personal
)


In [ ]:
client.fine_tuning.jobs.retrieve(finetune_job.id)

In [ ]:
"ft:gpt-4o-mini-2024-07-18:personal::AKeUzM35"


## use the finetuned model: generate json

In [ ]:
unique_layers = adata.obs[name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

cell_names_mapping ={'Astro': 'Astrocyte',
 'EndoMural': 'Endothelial and mural cells',
 'Excit_L2_3': 'Excitatory neuron layer 2/3',
 'Excit_L3': 'Excitatory neuron layer 3',
 'Excit_L3_4_5': 'Excitatory neuron layer 3/4/5',
 'Excit_L4': 'Excitatory neuron layer 4',
 'Excit_L5': 'Excitatory neuron layer 5',
 'Excit_L5_6': 'Excitatory neuron layer 5/6',
 'Excit_L6': 'Excitatory neuron layer 6',
 'Inhib': 'Inhibitory neuron',
 'Micro': 'Microglia',
 'OPC': 'Oligodendrocyte precursor cell',
 'Oligo': 'Oligodendrocyte'}


config = load_config('configs/config_151509_finetune.yaml')
config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping
config.system_prompt = prompt.finetune_system_deconv(config)

In [ ]:
prompt.finetune_user_deconv(val_neighbor_normalized_df, [0], config)

In [ ]:
generate_json_end2end(val_neighbor_normalized_df, config, prompt_func=prompt.finetune_user_deconv, n_rows=1, batch_size=5000)

In [ ]:
# submit the json

In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 1
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=extract_dict.keys())], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['finetune_gpt4o']


In [ ]:
gpt_results_df.index.difference(val_neighbor_normalized_df.index)


In [ ]:
gpt_results_df.finetune_gpt4o.value_counts()

In [ ]:
gpt_results_df.loc[gpt_results_df['finetune_gpt4o'] =='4', 'finetune_gpt4o'] = "Layer4"

## plot

In [ ]:
val_adata = adata[val_neighbor_normalized_df.index].copy()
val_adata.obs = val_adata.obs.join(gpt_results_df)
sc.pl.spatial(val_adata, color=['finetune_gpt4o', name_truth], library_id=data_name, size=1.6)

print(adjusted_rand_score(val_adata.obs[name_truth], val_adata.obs['finetune_gpt4o']))

## test on other data

In [ ]:
config = load_config('configs/config_151507_finetune_test.yaml')

In [ ]:
data_name = config.data_name
name_truth = config.name_truth
# --- Load data ---
data_path = str(dataset_dir("visium_libd", data_name))
# marker_path = "data/reference/marker_genes/Mouse_cell_markers.txt"
adata = sc.read_visium(data_path)
adata.var_names_make_unique()

# Normalize data
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)

cell_proportion_data = pd.read_csv(os.path.join(data_path, f"celltype_proportions_{data_name}.csv"), index_col=0)

adata.obs = adata.obs.join(cell_proportion_data)

# read the metadata
meta_data = pd.read_csv(os.path.join(data_path, "metadata.tsv"), sep="\t")
# merge the metadata to adata
adata.obs = adata.obs.merge(meta_data, left_index=True, right_index=True, how="left")

# Remove rows with NaN values in 'layer_guess'
adata = adata[~adata.obs[name_truth].isna()].copy()

# Verify that NaNs have been removed
remaining_nan_count = adata.obs[name_truth].isna().sum()
print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

# Rename the obs_names of adata
adata.obs_names = [f'spot_{i}' for i in range(len(adata.obs_names))]
# Transform spatial coordinates to DataFrame for sparse_adjacency
pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
cell_proportion_data = adata.obs[cell_proportion_data.columns].copy()

adj_matrix, distances = sparse_adjacency(pos_data, threshold=config.r, add_diagonal=True)

# Calculate the number of neighbors of each node
n_neighbors = adj_matrix.sum(axis=1).mean()
print(f"Number of neighbors: {sig_figs(n_neighbors, 3)}")

# If you want to store the n_neighbors of each node
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
adata.obs['n_neighbors'] = n_neighbors


# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(cell_proportion_data)


# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

# transform the neighbor matrix to a dataframe
neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized, index=cell_proportion_data.index, 
                              columns=cell_proportion_data.columns)




In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate mean cell type proportions for each group
mean_proportions = cell_proportion_data.groupby(adata.obs[name_truth]).mean()


# Create a stacked bar plot
fig, ax = plt.subplots(figsize=(10, 8))
mean_proportions.plot(kind='bar', stacked=True, ax=ax)

# Customize the plot
plt.title('Cell Type Proportions by Layer', fontsize=16)
plt.xlabel('Layer', fontsize=12)
plt.ylabel('Mean Proportion', fontsize=12)
plt.legend(title='Cell Type', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=14)
plt.tight_layout()

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')

# Add percentage labels on the bars
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f', label_type='center', fontsize=8)

plt.show()

# Display the mean proportions as a table
print(mean_proportions.to_string())

In [ ]:
unique_layers = adata.obs[name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

cell_names_mapping ={'Astro': 'Astrocyte',
 'EndoMural': 'Endothelial and mural cells',
 'Excit_L2_3': 'Excitatory neuron layer 2/3',
 'Excit_L3': 'Excitatory neuron layer 3',
 'Excit_L3_4_5': 'Excitatory neuron layer 3/4/5',
 'Excit_L4': 'Excitatory neuron layer 4',
 'Excit_L5': 'Excitatory neuron layer 5',
 'Excit_L5_6': 'Excitatory neuron layer 5/6',
 'Excit_L6': 'Excitatory neuron layer 6',
 'Inhib': 'Inhibitory neuron',
 'Micro': 'Microglia',
 'OPC': 'Oligodendrocyte precursor cell',
 'Oligo': 'Oligodendrocyte'}



config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping
config.system_prompt = prompt.finetune_system_deconv(config)

generate_json_end2end(neighbor_normalized_df, config, prompt_func=prompt.finetune_user_deconv, n_rows=1, batch_size=5000)

In [ ]:
# submit the json

In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 1
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=extract_dict.keys())], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['finetune_gpt4o']

In [ ]:
gpt_results_df.index.difference(neighbor_normalized_df.index)



In [ ]:
gpt_results_df.loc[gpt_results_df['finetune_gpt4o'] =='Layer7', 'finetune_gpt4o'] = "unknown"

In [ ]:
gpt_results_df.finetune_gpt4o.value_counts()

In [ ]:
adata.obs = adata.obs.join(gpt_results_df)
# replace the NA in adata.obs['finetune_gpt4o'] with "unknown"
adata.obs['finetune_gpt4o'] = adata.obs['finetune_gpt4o'].fillna("unknown")

sc.pl.spatial(adata, color=['finetune_gpt4o', name_truth], library_id=data_name, size=1.4)
print(adjusted_rand_score(adata.obs[name_truth], adata.obs['finetune_gpt4o']))


In [ ]:
# check the result of Layer 7
adata.obs.loc[adata.obs['finetune_gpt4o'] == "Layer7", ]
i = np.where(adata.obs['finetune_gpt4o'] == "Layer7")[0][0]
# system prompt
system_p = prompt.finetune_system_deconv(config)
# user prompt
user_p = prompt.finetune_user_deconv(neighbor_normalized_df, i, config)
# assistant prompt
assistant_p = prompt.finetune_assistant(neighbor_normalized_df, i, adata.obs[name_truth])
print(system_p)
print(user_p)
print(assistant_p)

In [ ]:
adata.obs.loc[adata.obs['finetune_gpt4o'] == "Layer7", name_truth]

In [ ]:
test_adata = sc.read_h5ad(str(dataset_file("merfish", "MERFISH_25")))